# Comparación reproducible de 3 modelos Hugging Face — SST-2
**Actividad:** evaluación de modelos preentrenados de NLP para clasificación binaria de opiniones cinematográficas en **inglés**.

**Salida:** tablas de `accuracy`, `precision`, `recall`, `F1-score`, latencia end-to-end, memoria GPU, decisiones técnicas y archivos CSV/JSON exportables.

> **Importante:** este notebook contiene código ejecutable, pero las celdas de resultados quedan sin ejecutar hasta abrirlo en Google Colab. **No se inventan resultados.**

**Antes de ejecutar:** en Google Colab, seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (o la GPU disponible), guarde una copia en Drive y ejecute todas las celdas en orden. Requiere conexión a Internet. Los modelos son públicos; el token de HF es opcional.

## 1. Ecosistema, caso y protocolo
**Hugging Face Models** aloja checkpoints, tokenizadores, revisiones y licencias; **Datasets** distribuye particiones y etiquetas para evaluación; **Spaces** ofrece demostraciones y despliegues interactivos (no se publica ningún Space en este experimento).

**Caso de uso:** clasificar opiniones de reseñas cinematográficas como *negativas (0)* o *positivas (1)*. Usamos la división **validation de `nyu-mll/glue`, configuración `sst2` (872 registros)**. Esta prueba no acredita rendimiento en español ni en servicios bancarios.

**Criterios:** compatibilidad con clases binarias, documentación de licencia, F1 macro y accuracy, recall de la clase positiva, latencia p95 individual, velocidad por lotes, pico de VRAM, costo aproximado en segundos GPU. Se evalúan los **mismos textos, etiquetas, hardware, lote 32, truncamiento a 128 tokens y tres pasadas de calentamiento**; no se ajustan pesos para no contaminar la validación.

## 2. Preparación del entorno y dependencias
Se utiliza PyTorch ya instalado por Colab para preservar su instalación CUDA. Si cambian paquetes incompatibles, reinicie el entorno y ejecute de nuevo.

In [ ]:
%pip -q install "transformers==4.52.4" "datasets==3.6.0" "huggingface_hub==0.33.0" "accelerate==1.7.0" "scikit-learn==1.6.1" "pandas==2.2.3" "numpy>=1.26,<3" "safetensors>=0.5.3,<1" "matplotlib>=3.8,<4"

In [ ]:
import os, random, gc, time, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
os.environ["TRANSFORMERS_NO_TF"] = "1"
import numpy as np
import pandas as pd
import torch
import transformers, datasets
from packaging.version import Version
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
from huggingface_hub import HfApi, login
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Active GPU en Colab: Entorno de ejecución > Cambiar tipo > T4 GPU."
assert Version(torch.__version__.split('+')[0]) >= Version('2.6'), "Se requiere torch>=2.6 para checkpoints con .bin. Actualice PyTorch CUDA en Colab."
DEVICE = 0
GPU = torch.cuda.get_device_name(0)
BATCH_SIZE = 32
MAX_LENGTH = 128
DATA_DIR = Path('data'); RESULTS_DIR = Path('results')
DATA_DIR.mkdir(exist_ok=True); RESULTS_DIR.mkdir(exist_ok=True)
print('GPU:', GPU, '| CUDA:', torch.version.cuda, '| torch:',torch.__version__)
print('transformers:', transformers.__version__,'| datasets:',datasets.__version__)
print('Python:',platform.python_version())

## 3. Autenticación segura a Hugging Face Hub
El acceso a estos repositorios públicos funciona **sin token**, pero se demuestra autenticación opcional. Para iniciar sesión, cree un token de lectura en Hugging Face y guárdelo en **Colab → Secrets → `HF_TOKEN`**. No lo escriba en el código ni en GitHub.

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except (ImportError, KeyError, Exception):
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Autenticación al Hub realizada mediante secreto (token oculto).')
else:
    print('Sin token: se utilizará acceso anónimo a recursos públicos del Hub.')
api = HfApi()

## 4. Dataset, control del esquema y registro de la fuente
Descargamos la revisión actual de `nyu-mll/glue` y guardamos el **SHA exacto** usado. Para replicar un experimento de otro día, fije `DATASET_REVISION` con la revisión registrada en `results/environment.json`.

Etiquetas SST-2: **0 negativo, 1 positivo**. **No se usa `test`** porque sus etiquetas no son públicas. El CSV exportado se puede abrir con Excel.

In [ ]:
DATASET_ID = 'nyu-mll/glue'
DATASET_CONFIG = 'sst2'
DATASET_REVISION = None  # Opcional: pegar aquí el SHA guardado en una ejecución anterior.
actual_dataset_revision = DATASET_REVISION or api.dataset_info(DATASET_ID).sha
raw = load_dataset(DATASET_ID, DATASET_CONFIG, split='validation', revision=actual_dataset_revision)
assert len(raw) == 872, f'Esperábamos 872 registros; obtenidos {len(raw)}.'
df = raw.to_pandas()[['idx', 'sentence', 'label']].copy()
assert df['idx'].is_unique and df['label'].isin([0,1]).all() and df['sentence'].notna().all()
df['source'] = 'nyu-mll/glue:sst2';df['split'] = 'validation'
DATA_CSV = DATA_DIR/'sst2_validation.csv'
df.to_csv(DATA_CSV, index=False, encoding='utf-8')
DATA_SHA = hashlib.sha256(DATA_CSV.read_bytes()).hexdigest()
metadata = {'dataset_id':DATASET_ID,'configuration':DATASET_CONFIG,'split':'validation',
            'revision':actual_dataset_revision,'n':int(len(df)),'sha256':DATA_SHA,
            'url':'https://huggingface.co/datasets/nyu-mll/glue'}
(DATA_DIR/'sst2_validation.metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Filas:',len(df),'| conteo por clase:',df['label'].value_counts().to_dict())
print('Revisión:',actual_dataset_revision,'| SHA256 CSV:',DATA_SHA)
display(df.head(6))

## 5. Modelos, selección y licencias
Los tres checkpoints fueron afinados para SST-2. Se comparan **DistilBERT (~67 M parámetros)**, **BERT base (~110 M)** y **RoBERTa base (~125 M)**. El conteo real de parámetros se extrae del modelo cargado. Según la información pública consultada, la ficha de DistilBERT indica **Apache-2.0**; las fichas de los checkpoints de TextAttack **no declaran una licencia clara**. Su inclusión en la comparación técnica **no implica autorización para desplegarlos comercialmente**.

> Los checkpoints muestran etiquetas `NEGATIVE`/`POSITIVE` o `LABEL_0`/`LABEL_1`; el mapeo a enteros de SST-2 es explícito y se valida antes de inferir.

In [ ]:
MODELS = {
    'DistilBERT': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
    'BERT': 'textattack/bert-base-uncased-SST-2',
    'RoBERTa': 'textattack/roberta-base-SST-2',
}
LICENSE_STATUS = {'DistilBERT':'Apache-2.0 (declarada)',
                  'BERT':'No declarada en ficha del checkpoint',
                  'RoBERTa':'No declarada en ficha del checkpoint'}
# Dejar una revisión exacta cuando se desee repetir un experimento anterior:
MODEL_REVISIONS = {'DistilBERT': None, 'BERT': None, 'RoBERTa': None}
model_catalog = pd.DataFrame([{'modelo':k,'hf_id':v,'licencia':LICENSE_STATUS[k]} for k,v in MODELS.items()])
display(model_catalog)

## 6. Funciones de inferencia y medición
La latencia *batch 32* se reporta en milisegundos amortizados por texto; la latencia p50/p95 mide **una solicitud a la vez** sobre 80 textos (o menos si el dataset cambia). Cronometramos **tokenización + forward + posprocesado** con `torch.cuda.synchronize()` antes y después. Las descargas y el tiempo de carga **no** forman parte de la latencia de inferencia.

In [ ]:
def label_to_id(name):
    v = str(name).strip().upper()
    if v in ('NEGATIVE','LABEL_0','0'): return 0
    if v in ('POSITIVE','LABEL_1','1'): return 1
    raise ValueError(f'Etiqueta desconocida: {name}. Revise el mapeo de clases.')

def cuda_sync():
    torch.cuda.synchronize()

def latency_one_by_one(clf, texts, n=80):
    samples = texts[:min(n,len(texts))]
    for text in samples[:5]:
        clf(text,truncation=True,max_length=MAX_LENGTH)
    elapsed = []
    for text in samples:
        cuda_sync()
        start=time.perf_counter()
        clf(text,truncation=True,max_length=MAX_LENGTH)
        cuda_sync()
        elapsed.append((time.perf_counter()-start)*1000)
    return float(np.percentile(elapsed,50)),float(np.percentile(elapsed,95))

TEXTS = df['sentence'].astype(str).tolist()
TRUE_LABELS = df['label'].astype(int).tolist()
print('Clases esperadas: 0 = negativo, 1 = positivo; casos:',len(TEXTS))

## 7. Ejecución de inferencia en los 3 modelos
El código descarga, evalúa y libera los checkpoints **secuencialmente** para que quepan en una GPU gratuita. Registra también la revisión precisa de cada modelo, de modo que los resultados sean auditables.

In [ ]:
rows = []
all_predictions = []
for model_name, model_id in MODELS.items():
    print('\n' + '='*70 + '\nModelo:',model_name,'|',model_id)
    gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    revision = MODEL_REVISIONS[model_name] or api.model_info(model_id).sha
    tokenizer = AutoTokenizer.from_pretrained(model_id,revision=revision)
    model = AutoModelForSequenceClassification.from_pretrained(model_id,revision=revision)
    assert model.config.num_labels == 2, 'Checkpoint incompatible con clasificación binaria.'
    label_map = {int(k):label_to_id(v) for k,v in model.config.id2label.items()}
    assert sorted(label_map.values()) == [0,1],f'Mapeo incoherente: {label_map}'
    clf = pipeline('text-classification',model=model,tokenizer=tokenizer,device=DEVICE,framework='pt')
    for _ in range(3):
        clf(TEXTS[:BATCH_SIZE],batch_size=BATCH_SIZE,truncation=True,max_length=MAX_LENGTH)
    cuda_sync();start=time.perf_counter()
    outputs = clf(TEXTS,batch_size=BATCH_SIZE,truncation=True,max_length=MAX_LENGTH)
    cuda_sync();seconds=time.perf_counter()-start
    preds = [label_to_id(o['label']) for o in outputs]
    assert len(preds)==len(TRUE_LABELS)
    acc=accuracy_score(TRUE_LABELS,preds)
    p,r,f,_=precision_recall_fscore_support(TRUE_LABELS,preds,average='macro',zero_division=0)
    pp,rr,ff,_=precision_recall_fscore_support(TRUE_LABELS,preds,average='binary',pos_label=1,zero_division=0)
    matrix=confusion_matrix(TRUE_LABELS,preds,labels=[0,1])
    lat50,lat95=latency_one_by_one(clf,TEXTS)
    peak_mib=torch.cuda.max_memory_allocated()/1024**2
    row={'modelo':model_name,'hf_id':model_id,'revision':revision,
        'licencia_declarada':LICENSE_STATUS[model_name], 'n':len(TRUE_LABELS),
        'accuracy':float(acc),'precision_macro':float(p),'recall_macro':float(r),'f1_macro':float(f),
        'precision_positiva':float(pp),'recall_positiva':float(rr),'f1_positiva':float(ff),
        'latencia_media_lote_ms_por_texto':1000*seconds/len(TRUE_LABELS),
        'throughput_textos_s':len(TRUE_LABELS)/seconds,
        'latencia_p50_individual_ms':lat50,'latencia_p95_individual_ms':lat95,
        'pico_vram_mib':peak_mib,'parametros_millones':clf.model.num_parameters()/1e6,
        'gpu_segundos_por_1000_textos':1000/(len(TRUE_LABELS)/seconds),
        'matriz_confusion':json.dumps(matrix.tolist())}
    rows.append(row)
    all_predictions.append(pd.DataFrame({'modelo':model_name,'idx':df['idx'].tolist(),
        'sentence':TEXTS,'label_real':TRUE_LABELS,
        'label_predicha':preds,'confianza_predicha':[o['score'] for o in outputs]}))
    print('Accuracy:',round(acc,4),'| F1 macro:',round(f,4),'| p95 individual ms:',round(lat95,2),
          '| VRAM pico MiB:',round(peak_mib,1))
    print('Matriz de confusión [real=0/1, predicho=0/1]:\n',matrix)
    del clf,model,tokenizer
    gc.collect();torch.cuda.empty_cache()
comparison = pd.DataFrame(rows)
prediction_df=pd.concat(all_predictions,ignore_index=True)
comparison.to_csv(RESULTS_DIR/'comparacion_modelos.csv',index=False)
prediction_df.to_csv(RESULTS_DIR/'predicciones.csv',index=False)
print('Se completaron los tres modelos.')

## 8. Tabla comparativa, gráficos y errores
**Métricas principales:** precision/recall/F1 **macro** tratan ambas clases por igual; también se reportan precision, recall y F1 de la **clase positiva**. Latencia p95 se estima con 80 predicciones individuales y cambia con hardware/carga.

In [ ]:
columns=['modelo','parametros_millones','licencia_declarada','accuracy','precision_macro','recall_macro','f1_macro',
         'recall_positiva','latencia_media_lote_ms_por_texto','latencia_p95_individual_ms',
         'throughput_textos_s','pico_vram_mib','gpu_segundos_por_1000_textos']
display(comparison[columns].round(3).sort_values('f1_macro',ascending=False))
fig,ax=plt.subplots(figsize=(8,4))
comparison.set_index('modelo')[['accuracy','precision_macro','recall_macro','f1_macro']].plot.bar(ax=ax,ylim=(0,1))
ax.set_title('Métricas de clasificación en SST-2 validation');ax.set_ylabel('Puntuación');ax.tick_params(axis='x',rotation=0)
plt.tight_layout();plt.savefig(RESULTS_DIR/'metricas.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(comparison['latencia_p95_individual_ms'],comparison['f1_macro'],s=100)
for _,x in comparison.iterrows():ax.annotate(x['modelo'],(x['latencia_p95_individual_ms'],x['f1_macro']),xytext=(5,5),textcoords='offset points')
ax.set_xlabel('Latencia p95 individual (ms) — menor es mejor')
ax.set_ylabel('F1 macro — mayor es mejor')
ax.set_title('Trade-off: velocidad frente a desempeño')
plt.tight_layout();plt.savefig(RESULTS_DIR/'tradeoff.png',dpi=160);plt.show()
for name in MODELS:
    errors=prediction_df[(prediction_df.modelo==name)&(prediction_df.label_real!=prediction_df.label_predicha)]
    print(name,'| errores:',len(errors))
    display(errors[['idx','sentence','label_real','label_predicha','confianza_predicha']].head(4))

## 9. Umbrales, recomendación condicionada y riesgos
**Regla propuesta de aceptación** (reajustar según producto, antes de mirar resultados): licencia de checkpoint explícita compatible; `accuracy ≥ 0.89`, `F1 macro ≥ 0.89`, `recall positivo ≥ 0.88`, `p95 individual ≤ 80 ms` y `pico VRAM ≤ 4096 MiB`. Si ninguno cumple, **no se aprueba ninguno**. Los modelos TextAttack quedan *no elegibles para despliegue* hasta confirmar por escrito sus licencias; siguen siendo comparadores técnicos.

**Riesgos:** dominio cinematográfico, idioma inglés, sarcasmo, negaciones, diferencias entre GPU T4/L4, sesgo, drift, confianza mal calibrada y propiedad intelectual de datos y pesos.

In [ ]:
approved_license={'DistilBERT':True,'BERT':False,'RoBERTa':False}
selection=comparison.copy()
selection['licencia_aceptable']=selection['modelo'].map(approved_license)
selection['supera_umbrales']=(selection['licencia_aceptable'] &
    (selection['accuracy']>=0.89)&(selection['f1_macro']>=0.89)&
    (selection['recall_positiva']>=0.88)&
    (selection['latencia_p95_individual_ms']<=80)&
    (selection['pico_vram_mib']<=4096))
display(selection[['modelo','licencia_aceptable','supera_umbrales','f1_macro','latencia_p95_individual_ms']])
eligible=selection[selection['supera_umbrales']].sort_values(
    ['f1_macro','latencia_p95_individual_ms'],ascending=[False,True])
recommended = None if eligible.empty else eligible.iloc[0]['modelo']
if recommended:
    print('MODELO RECOMENDADO SEGÚN REGLAS:',recommended)
else:
    print('NINGÚN MODELO CUMPLE TODOS LOS CRITERIOS. No declarar un ganador listo para desplegar.')
conclusions=[
    '# Conclusiones experimentales',
    f'Dataset: {DATASET_ID}/{DATASET_CONFIG} — validation, {len(df)} registros.',
    f'GPU: {GPU}. Lote: {BATCH_SIZE}; max_length: {MAX_LENGTH}.',
    'Métricas macro y latencia registradas en results/comparacion_modelos.csv.',
    f'Modelo aprobado por umbrales y licencia: {recommended or "NINGUNO"}.',
    'DistilBERT parte con ventaja práctica por su licencia Apache-2.0 y menor tamaño;',
    'esta observación previa no sustituye la evaluación experimental.',
    'BERT/RoBERTa TextAttack no son elegibles para despliegue sin validar licencias.',
    'Evaluar nuevo corpus de dominio e idioma objetivo antes de puesta en producción.'
]
(RESULTS_DIR/'conclusiones.md').write_text('\n'.join(conclusions),encoding='utf-8')
print('\n'.join(conclusions))

## 10. Evidencias y exportación para GitHub
Se guardan las versiones de bibliotecas, hardware, SHA-256 del CSV, revisiones de los tres modelos y parámetros de medición. El archivo ZIP facilita anexar resultados reales al repositorio luego de ejecutar el Colab. No publique `HF_TOKEN`. El CSV experimental procede de SST-2 y está sujeto a la verificación de derechos de reutilización.

In [ ]:
environment = {
    'executed_utc':datetime.now(timezone.utc).isoformat(),
    'gpu':GPU,'cuda':torch.version.cuda,'torch':torch.__version__,
    'python':platform.python_version(),'transformers':transformers.__version__,
    'datasets':datasets.__version__,'numpy':np.__version__,'pandas':pd.__version__,
    'dataset':metadata,'model_revisions':{r['modelo']:r['revision'] for r in rows},
    'seed':SEED,'batch_size':BATCH_SIZE,'max_length':MAX_LENGTH,
    'latency_method':'end-to-end pipeline, CUDA sync, 3 warmups, 80 requests individual para p50/p95',
    'weights_finetuned_in_experiment':False,'benchmark_status':'real GPU run'
}
(RESULTS_DIR/'environment.json').write_text(json.dumps(environment,ensure_ascii=False,indent=2),encoding='utf-8')
import zipfile
out = Path('salidas_colab_sst2.zip')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for directory in (DATA_DIR,RESULTS_DIR):
        for f in directory.rglob('*'):
            if f.is_file():z.write(f,str(f))
print('Archivos generados:',[str(f) for f in sorted(RESULTS_DIR.glob('*'))])
print('Paquete:',out,round(out.stat().st_size/1024,1),'KiB')
# Descarga opcional en Colab:
# from google.colab import files
# files.download(str(out))

# Interpretación y entrega de resultados
El bloque de evaluación calcula **accuracy**, **precision macro**, **recall macro**, **F1 macro**, latencia p50/p95, throughput y VRAM; las predicciones y el entorno se exportan en un ZIP.

La recomendación se obtiene mediante los umbrales definidos: cuando solo DistilBERT tiene licencia documentada, la aprobación será DistilBERT **si supera todos los mínimos**. Si RoBERTa destaca en F1, eso no implica automáticamente que sea el más adecuado para un requisito de baja latencia o uso sujeto a licencia.

**Interpretación de reproducibilidad:** el informe de la entrega recoge una ejecución real en Tesla T4 de octubre de 2026. Nuevas sesiones y revisiones de checkpoints pueden producir medidas distintas. Consultar `results/environment.json` de la ejecución correspondiente.

**Precaución:** SST-2 proviene de reseñas de películas en inglés; no transferir sin validar idioma y dominio. Los archivos con texto de SST-2 requieren revisión de derechos antes de publicación abierta.
